# App crash reports per day: c chart

**c Control Chart** · a worked example from the StatNex teaching guide

**The story.** A shopping app's support team counts crash reports every day. On day 11 a buggy update went out.

**The question.** Is the number of crashes stable, and does the bad update stand out?

**To run it**, choose **Runtime → Run all**. Colab first warns that this notebook was not written by Google; it reads none of your data, so choose **Run anyway**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

In [ ]:
# Each script saves its figures to the folder figures/, as it would on your computer.
# This shows each new figure under the cell that saved it.
from pathlib import Path

import matplotlib
from IPython import get_ipython
from IPython.display import Image, display

matplotlib.use("Agg")
shown = set()


def show_new_figures(*_):
    for path in sorted(Path("figures").glob("*.png"), key=lambda p: p.stat().st_mtime_ns):
        if (path, path.stat().st_mtime_ns) not in shown:
            shown.add((path, path.stat().st_mtime_ns))
            display(Image(filename=str(path)))


events = get_ipython().events
for old in [f for f in events.callbacks["post_run_cell"] if f.__name__ == "show_new_figures"]:
    events.unregister("post_run_cell", old)
events.register("post_run_cell", show_new_figures)

## The table

This cell saves the table `10_control_chart/c_chart.csv` (25 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Day,Crash reports
1.0,13.0
2.0,8.0
3.0,9.0
4.0,6.0
5.0,10.0
6.0,11.0
7.0,7.0
8.0,8.0
9.0,6.0
10.0,7.0
11.0,21.0
12.0,11.0
13.0,7.0
14.0,12.0
15.0,6.0
16.0,13.0
17.0,10.0
18.0,13.0
19.0,7.0
20.0,8.0
21.0,5.0
22.0,9.0
23.0,8.0
24.0,8.0
25.0,6.0

These cells save `control_signals.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile control_signals.py
"""Reading a Shewhart control chart: the two pattern rules, and the chart itself.

SciPy and statsmodels have no control charts, so both are written out here.

A point outside the 3-sigma limits is the obvious signal. A chart is also read for its
shape, because two patterns inside the limits are about as unlikely as a point outside
when nothing has changed (Nelson's rules 2 and 3; ISO 7870-2 and JIS Z 9020-2 list the
same tests):

* 9 points in a row on one side of the centre line -- the level may have shifted;
* 6 rises in a row (7 points climbing), or 6 falls -- a trend.

These are the two StatNex applies. Each run is reported once, from its first point to
its last, however long it lasts: nine in a row sets the warning off, and a run of
eighteen is still one run. A point exactly on the centre line, or a missing one (the
first moving range), breaks a run.
"""

import numpy as np
from matplotlib.figure import Figure


def stretches(signs, minimum):
    """Each stretch of at least `minimum` equal, non-zero signs, as (first, last) positions."""
    found, start = [], 0
    for i in range(1, len(signs) + 1):
        if i == len(signs) or signs[i] != signs[start]:
            if signs[start] != 0 and i - start >= minimum:
                found.append((start, i - 1))
            start = i
    return found


def pattern_signals(label, values, centre):
    """Messages for the runs and trends in one chart's points (a pandas Series)."""
    numbers = list(values.index)
    points = values.to_numpy(float)
    found = []

    side = np.where(np.isfinite(points), np.sign(points - np.asarray(centre, float)), 0.0)
    for first, last in stretches(side, 9):
        where = "above" if side[first] > 0 else "below"
        found.append(f"{label}: points {numbers[first]} to {numbers[last]} are {last - first + 1} in a row "
                     f"{where} the centre line. The level of the process may have shifted.")

    # Step i joins points i - 1 and i; six steps one way are seven points climbing or falling.
    steps = np.zeros(len(points))
    for i in range(1, len(points)):
        if np.isfinite(points[i]) and np.isfinite(points[i - 1]):
            steps[i] = np.sign(points[i] - points[i - 1])
    for first, last in stretches(steps, 6):
        where = "rise" if steps[first] > 0 else "fall"
        found.append(f"{label}: points {numbers[first - 1]} to {numbers[last]} {where} "
                     f"monotonically. This is a sign of a change in trend.")
    return found


def draw(path, title, values, centre, upper, lower, xlabel="Subgroup"):
    """One chart: the points, the centre line, and the limits (stepped if they vary)."""
    fig = Figure(figsize=(7, 4.5))
    ax = fig.subplots()
    x = np.asarray(values.index)
    ax.plot(x, values.to_numpy(float), marker="o", color="black", linewidth=1)
    for line, style in ((upper, "--"), (centre, "-"), (lower, "--")):
        ax.step(x, np.broadcast_to(np.asarray(line, float), x.shape), where="mid",
                linestyle=style, color="grey", linewidth=1)
    out = (values > upper) | (values < lower)
    ax.plot(x[out.to_numpy()], values[out].to_numpy(float), "o", color="red")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    fig.savefig(path, dpi=150)

## Libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from control_signals import draw, pattern_signals

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

## The data

In [ ]:
# One row per inspection unit of the same size (a day, a sheet, a batch): the number of
# nonconformities (defects, errors, complaints) counted in it. Blanks, text and negative
# counts stop the chart.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
column = "Crash reports"
defects = pd.to_numeric(data[column], errors="coerce")
if defects.isna().any() or (defects < 0).any():
    raise ValueError("Every count must be a number of 0 or more.")
defects.index = np.arange(1, len(defects) + 1)  # subgroup numbers 1, 2, ...


def chart(counts):
    """Poisson limits: the variance equals the mean, so c-bar alone sets the width."""
    # Each count is treated as Poisson: defects that turn up independently, at a steady
    # average rate, in units of the same size. Then
    #   CL = c-bar,  UCL, LCL = c-bar +/- 3 sqrt(c-bar)
    # and a lower limit below 0 is drawn at 0. If the units differ in size, use the u chart.
    centre = counts.mean()
    margin = 3.0 * np.sqrt(centre)
    return pd.DataFrame({"c": counts, "CL": centre, "UCL": centre + margin,
                         "LCL": max(centre - margin, 0.0)})


def outside(table):
    return (table["c"] > table["UCL"]) | (table["c"] < table["LCL"])


counts = defects

## Control chart data

In [ ]:
# One row per unit: its count (c) against the centre line (CL) and the limits (UCL, LCL).
# A count outside the limits is a signal that something other than everyday variation
# affected that unit.
table = chart(counts)
print()
print(table)
print(f"\nSubgroups outside the limits: {list(table.index[outside(table)])}")

## The chart

In [ ]:
# Besides points outside the limits, a chart is read for patterns that are just as
# unlikely by chance: 9 points in a row on one side of the centre line, or 6 rises (or
# falls) in a row.
for message in pattern_signals("c chart", table["c"], table["CL"]):
    print(message)
Path("figures").mkdir(exist_ok=True)
draw("figures/c_chart.png", "c chart", table["c"], table["CL"], table["UCL"], table["LCL"])

## What the result says

A c chart follows a count per day. The centre line (CL) is the average count: 9.16 crash reports a day. The control limits mark how far the count can move by ordinary chance alone: from the lower control limit (LCL) of 0.08 up to the upper control limit (UCL) of 18.24.

Of the 25 days, only day 11 falls outside these limits, with 21 crash reports. That is above the UCL, so chance alone is a poor explanation. Day 11 is the day the buggy update went out.

The crash count is stable at about 9 reports a day, apart from that one spike. The update is the place to look, not the ordinary days.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.